# Mini caso 5 — Generación automática de resúmenes documentales

Este notebook implementa el mini caso 5 sobre el corpus RTVE 23-F con un alcance concreto: **generar resúmenes documentales a partir de `text_clean_base`**, manteniendo trazabilidad mediante `doc_id`, `title` y `pdf_url`.

La versión combina dos capas:

1. **Método híbrido reproducible obligatorio:** parte del `summary` original como resumen semilla, elimina su truncamiento final y lo completa con evidencias seleccionadas desde `text_clean_base`.
2. **Capa LLM opcional:** si se activa con variables de entorno, reescribe el resumen usando solo el resumen semilla y las evidencias extraídas del documento. Si no hay token, hay límites de uso o falla la API, se conserva automáticamente el método híbrido.

La idea no es depender de una API para que el notebook funcione, sino construir una salida completa para los 167 documentos y registrar explícitamente qué documentos se han resumido con LLM y cuáles han quedado cubiertos por fallback. En esta ejecución se intenta aplicar la capa LLM al corpus completo, manteniendo fallback para no romper la reproducibilidad.

## 0. Configuración reproducible y uso opcional de LLM

Todas las rutas se resuelven desde la raíz del repositorio.

La capa LLM está desactivada por defecto. Para activarla desde terminal antes de abrir Jupyter:

### GitHub Models

```bash
export ENABLE_LLM_SUMMARIES=1
export GITHUB_MODELS_TOKEN="TU_TOKEN"
export GITHUB_MODELS_MODEL="openai/gpt-4.1-mini"
export LLM_MAX_DOCUMENTS=167
export LLM_SELECTION_STRATEGY=first
```

Para una prueba rápida, `LLM_MAX_DOCUMENTS` puede bajarse a 20. Para la ejecución final de este notebook se ha intentado usar `LLM_MAX_DOCUMENTS=167`, de forma que se prueba la capa generativa sobre todo el corpus y se deja que el fallback cubra los errores de API.

El token no debe escribirse en el notebook ni subirse al repositorio.

In [18]:
from pathlib import Path
from collections import Counter
import json
import math
import os
import re
import time
import urllib.error
import urllib.request
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Parámetros del método híbrido obligatorio
SHORT_SUMMARY_WORDS = 90
EXTENDED_SUMMARY_WORDS = 230
N_EVIDENCE_SENTENCES = 7
MIN_SENTENCE_WORDS = 8

# Parámetros de la capa LLM opcional
ENABLE_LLM_SUMMARIES = os.getenv("ENABLE_LLM_SUMMARIES", "0") == "1"
LLM_PROVIDER = "github"
LLM_MAX_DOCUMENTS = int(os.getenv("LLM_MAX_DOCUMENTS", "20"))
LLM_SLEEP_SECONDS = float(os.getenv("LLM_SLEEP_SECONDS", "0.7"))
LLM_SELECTION_STRATEGY = os.getenv("LLM_SELECTION_STRATEGY", "sample_by_length").strip().lower()

# GitHub Models
GITHUB_MODELS_TOKEN = os.getenv("GITHUB_MODELS_TOKEN") or os.getenv("GITHUB_TOKEN")
GITHUB_MODELS_MODEL = os.getenv("GITHUB_MODELS_MODEL", "openai/gpt-4.1-mini")
GITHUB_MODELS_ENDPOINT = os.getenv(
    "GITHUB_MODELS_ENDPOINT",
    "https://models.github.ai/inference/chat/completions",
)

LLM_TOKEN = GITHUB_MODELS_TOKEN
LLM_MODEL = GITHUB_MODELS_MODEL
LLM_ENDPOINT = GITHUB_MODELS_ENDPOINT
LLM_AVAILABLE = bool(ENABLE_LLM_SUMMARIES and LLM_TOKEN)

print("Configuración cargada")
print(f"LLM activado explícitamente: {ENABLE_LLM_SUMMARIES}")
print(f"Proveedor LLM: {LLM_PROVIDER}")
print(f"Token disponible en variable de entorno: {bool(LLM_TOKEN)}")
print(f"Capa LLM disponible para intentar uso: {LLM_AVAILABLE}")
print(f"Modelo configurado: {LLM_MODEL}")
print(f"Máximo de documentos con intento LLM: {LLM_MAX_DOCUMENTS}")
print(f"Estrategia de selección LLM: {LLM_SELECTION_STRATEGY}")

Configuración cargada
LLM activado explícitamente: True
Proveedor LLM: github
Token disponible en variable de entorno: True
Capa LLM disponible para intentar uso: True
Modelo configurado: openai/gpt-4.1-mini
Máximo de documentos con intento LLM: 167
Estrategia de selección LLM: first


**Lectura del bloque.**  
La configuración confirma que la capa LLM está activada, que el token está disponible como variable de entorno y que el proveedor es GitHub Models. En esta ejecución se ha configurado `LLM_MAX_DOCUMENTS=167` con estrategia `first`, por lo que el notebook intentará reescribir los 167 documentos del corpus. Aun así, el pipeline no depende de que todas las llamadas funcionen: cualquier error de API se resuelve mediante fallback híbrido.

## 1. Resolución de rutas y carga del corpus

Se busca la raíz del repositorio subiendo desde el directorio actual hasta encontrar `data/processed/rtve_corpus_clean_base.csv`.

In [2]:
def find_project_root(start_path: Path) -> Path:
    """Busca la raíz del proyecto a partir del archivo de entrada esperado."""
    start_path = start_path.resolve()
    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        expected = candidate / "data" / "processed" / "rtve_corpus_clean_base.csv"
        if expected.exists():
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado data/processed/rtve_corpus_clean_base.csv. "
        "Ejecuta el notebook desde dentro del repositorio o genera antes el corpus limpio."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rtve_corpus_clean_base.csv"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(f"Dataset cargado desde: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape[0]} documentos x {df.shape[1]} columnas")
display(df.head(3))

Dataset cargado desde: data/processed/rtve_corpus_clean_base.csv
Shape: 167 documentos x 25 columnas


,doc_id,source_document_id,title,pages,detail_url,pdf_url,summary,keywords,text_full,text_clean_base,text_length_chars,text_length_words,text_clean_length_chars,text_clean_length_words,moncloa_id,moncloa_section,moncloa_subsection,final_match_status,coverage_moncloa,alpha_ratio,digit_ratio,uppercase_ratio,weird_char_ratio,n_title_years,title_main_year
0,rtve_1860,1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,3,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_20_de_febrero_de_1982.pdf,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y cert...",C/SG/2820/20-02-82 DTOR. Vista oral 2/81,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82\n\n- Solo ha tenido lugar la sesión de la ma...,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82\n\n- Solo ha tenido lugar la sesión de la ma...,3934,640,3934,640,moncloa_0099,defensa,cni,high_confidence_match,True,0.777834,0.013726,0.147386,0.000000,1,1982.0
1,rtve_1859,1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,4,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_22_de_febrero_de_1982.pdf,"Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el Consejo Supremo de Justicia Militar, dond...",C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 22-02-82.\n\n- A...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 22-02-82.\n\n- A...,6417,1018,6417,1018,moncloa_0098,defensa,cni,high_confidence_match,True,0.781985,0.009506,0.195895,0.000156,1,1982.0
2,rtve_1858,1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,5,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_24_de_febrero_de_1982.pdf,"Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Justicia Militar en febrero de 1982, marca...",C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 23.02.82\n\nA...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 23.02.82\n\nA...,8183,1347,8183,1347,moncloa_0097,defensa,cni,high_confidence_match,True,0.784920,0.011487,0.124085,0.000611,1,1982.0


**Lectura del bloque.**  
El corpus se carga correctamente desde la estructura portable del repositorio.

## 2. Selección de campos y validación mínima

El mini caso solo necesita identificación, trazabilidad, el resumen original y el texto limpio. No se usan variables generadas por otros mini casos.

In [3]:
required_columns = ["doc_id", "title", "summary", "text_clean_base", "pdf_url"]
optional_columns = ["text_clean_length_words", "moncloa_section"]

missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(f"Faltan columnas obligatorias: {missing_required}")

available_columns = required_columns + [c for c in optional_columns if c in df.columns]
df_case5 = df[available_columns].copy()

for col in ["doc_id", "title", "summary", "text_clean_base", "pdf_url"]:
    df_case5[col] = df_case5[col].fillna("").astype(str)

if "text_clean_length_words" not in df_case5.columns:
    df_case5["text_clean_length_words"] = df_case5["text_clean_base"].str.split().str.len()
else:
    fallback_lengths = df_case5["text_clean_base"].str.split().str.len()
    df_case5["text_clean_length_words"] = (
        pd.to_numeric(df_case5["text_clean_length_words"], errors="coerce")
        .fillna(fallback_lengths)
        .astype(int)
    )

if "moncloa_section" not in df_case5.columns:
    df_case5["moncloa_section"] = ""
else:
    df_case5["moncloa_section"] = df_case5["moncloa_section"].fillna("").astype(str)

validation = {
    "n_documents": len(df_case5),
    "n_doc_id_unique": df_case5["doc_id"].nunique(),
    "n_empty_text_clean_base": int((df_case5["text_clean_base"].str.strip() == "").sum()),
    "min_text_clean_words": int(df_case5["text_clean_length_words"].min()),
    "median_text_clean_words": float(df_case5["text_clean_length_words"].median()),
    "max_text_clean_words": int(df_case5["text_clean_length_words"].max()),
}

pd.DataFrame([validation])

,n_documents,n_doc_id_unique,n_empty_text_clean_base,min_text_clean_words,median_text_clean_words,max_text_clean_words
0,167,167,0,72,579.0,95293


**Lectura del bloque.**  
La validación comprueba que existe una fila por documento y que `text_clean_base` está disponible. Si hubiera textos vacíos, quedarían identificados antes de generar resúmenes.

## 3. Diagnóstico del resumen original

El campo `summary` se usa como semilla, no como resultado final. Primero se comprueba si está sistemáticamente truncado.

In [4]:
df_case5["summary_length_chars"] = df_case5["summary"].str.len()
df_case5["summary_length_words"] = df_case5["summary"].str.split().str.len()
df_case5["summary_ends_with_ellipsis"] = df_case5["summary"].str.strip().str.endswith("...")

summary_diagnosis = {
    "n_documents": len(df_case5),
    "n_summary_available": int(df_case5["summary"].str.strip().ne("").sum()),
    "n_empty_summary": int(df_case5["summary"].str.strip().eq("").sum()),
    "min_summary_chars": int(df_case5["summary_length_chars"].min()),
    "median_summary_chars": float(df_case5["summary_length_chars"].median()),
    "max_summary_chars": int(df_case5["summary_length_chars"].max()),
    "n_summary_ends_with_ellipsis": int(df_case5["summary_ends_with_ellipsis"].sum()),
    "median_summary_words": float(df_case5["summary_length_words"].median()),
}

pd.DataFrame([summary_diagnosis])

,n_documents,n_summary_available,n_empty_summary,min_summary_chars,median_summary_chars,max_summary_chars,n_summary_ends_with_ellipsis,median_summary_words
0,167,167,0,303,303.0,303,167,49.0


**Lectura del bloque.**  
Se confirma que `n_summary_ends_with_ellipsis = 167`, por tanto todos los resúmenes terminan en puntos suspensivos, y en consecuencia no deben considerarse como resúmenes completos. Aun así, son útiles como semilla temática para orientar la selección de evidencias y la posterior reescritura con LLM.

## 4. Funciones de limpieza, selección de evidencias y resumen híbrido

El método híbrido no intenta "inventar" un resumen nuevo desde cero. Parte de `summary`, elimina el truncamiento y añade frases relevantes extraídas de `text_clean_base`. Esto mejora la trazabilidad, su limitación es que no reescribe realmente el contenido y por tanto el texto puede seguir siendo menos natural; por eso la capa LLM opcional puede aportar mejora clara de legibilidad.

In [5]:
SPANISH_STOPWORDS = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "como", "con", "contra",
    "cual", "cuando", "de", "del", "desde", "donde", "durante", "e", "el", "ella",
    "ellas", "ellos", "en", "entre", "era", "eran", "eras", "eres", "es", "esa",
    "esas", "ese", "eso", "esos", "esta", "estaba", "estaban", "estas", "este",
    "esto", "estos", "fue", "fueron", "ha", "han", "hasta", "hay", "la", "las",
    "le", "les", "lo", "los", "me", "mi", "mis", "muy", "más", "no", "nos", "o",
    "otra", "otras", "otro", "otros", "para", "pero", "por", "porque", "que", "se",
    "sin", "sobre", "son", "su", "sus", "también", "tanto", "te", "tiene", "tienen",
    "todo", "todos", "un", "una", "uno", "unos", "y", "ya",
}

WORD_RE = re.compile(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]{3,}")
SENTENCE_SPLIT_RE = re.compile(r"(?<=[\.\?\!])\s+|\n+")


def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def clean_summary_seed(summary):
    """Limpia el resumen original para usarlo como semilla temática."""
    s = normalize_spaces(summary)
    s = re.sub(r"^Resumen global del documento\s*:\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^Resumen global del documento sobre .*?:\s*", "", s, flags=re.IGNORECASE)
    s = s.rstrip()
    if s.endswith("..."):
        s = s[:-3].rstrip(" ,;:-")
    return normalize_spaces(s)


def tokenize_words(text):
    return [w.lower() for w in WORD_RE.findall(str(text))]


def limit_words(text, max_words):
    words = normalize_spaces(text).split()
    if len(words) <= max_words:
        return normalize_spaces(text)
    return " ".join(words[:max_words]).rstrip(" ,;:") + "..."


def split_sentences(text, min_words=MIN_SENTENCE_WORDS):
    raw_parts = SENTENCE_SPLIT_RE.split(str(text))
    sentences = []
    seen = set()

    for part in raw_parts:
        s = normalize_spaces(part)
        if not s:
            continue

        words = s.split()
        if len(words) < min_words:
            continue

        # Filtros simples contra tablas y ruido OCR.
        if s.count("|") >= 3:
            continue
        if len(s) > 900:
            continue
        if sum(ch.isdigit() for ch in s) / max(len(s), 1) > 0.35:
            continue
        if len(re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", s)) / max(len(s), 1) < 0.45:
            continue

        key = s.lower()
        if key in seen:
            continue
        seen.add(key)
        sentences.append(s)

    return sentences


def build_query_terms(title, summary_seed):
    source = f"{title} {summary_seed}"
    tokens = [
        t for t in tokenize_words(source)
        if t not in SPANISH_STOPWORDS and len(t) >= 4
    ]
    return Counter(tokens)


def sentence_score(sentence, query_terms, global_profile):
    tokens = [
        t for t in tokenize_words(sentence)
        if t not in SPANISH_STOPWORDS and len(t) >= 4
    ]
    if not tokens:
        return 0.0

    query_overlap = sum(query_terms.get(t, 0) for t in tokens)
    global_signal = sum(global_profile.get(t, 0.0) for t in tokens)

    # Penaliza frases que parecen encabezados o listas sueltas.
    starts_badly = 1 if re.match(r"^[-–—\d\.\)\s]+", sentence) else 0
    penalty = 0.85 if starts_badly else 1.0

    return penalty * (2.0 * query_overlap + 0.5 * global_signal) / math.sqrt(len(tokens))


def select_evidence_sentences(text, title, summary_seed, n_sentences=N_EVIDENCE_SENTENCES):
    sentences = split_sentences(text)

    if not sentences:
        return []

    query_terms = build_query_terms(title, summary_seed)
    all_tokens = [
        t for t in tokenize_words(text)
        if t not in SPANISH_STOPWORDS and len(t) >= 4
    ]
    counts = Counter(all_tokens)
    max_count = max(counts.values()) if counts else 1
    global_profile = Counter({term: freq / max_count for term, freq in counts.items()})

    scored = [
        (idx, sentence, sentence_score(sentence, query_terms, global_profile))
        for idx, sentence in enumerate(sentences)
    ]

    selected = sorted(scored, key=lambda x: x[2], reverse=True)[:n_sentences]
    selected = sorted(selected, key=lambda x: x[0])
    return [sentence for _, sentence, score in selected if score > 0]


def build_hybrid_summary(row):
    title = row["title"]
    original_summary = row["summary"]
    text = row["text_clean_base"]

    summary_seed = clean_summary_seed(original_summary)
    evidence = select_evidence_sentences(text, title, summary_seed)

    if not summary_seed and evidence:
        summary_seed = evidence[0]

    evidence_text = " ".join(evidence)

    resumen_corto = limit_words(summary_seed, SHORT_SUMMARY_WORDS)

    if evidence_text:
        resumen_extendido = normalize_spaces(
            summary_seed + " " + evidence_text
        )
    else:
        resumen_extendido = summary_seed

    resumen_extendido = limit_words(resumen_extendido, EXTENDED_SUMMARY_WORDS)

    return {
        "summary_seed_clean": summary_seed,
        "resumen_corto": resumen_corto,
        "resumen_extendido": resumen_extendido,
        "summary_method": "hybrid_summary_seed_plus_text_evidence",
        "summary_status": "ok_hibrido_sin_llm",
        "summary_error": "",
        "n_evidence_sentences": len(evidence),
        "evidence_sentences_json": json.dumps(evidence, ensure_ascii=False),
    }

## 5. Generación del método híbrido para todos los documentos

Se crea una primera salida completa. Esta tabla actúa como fallback obligatorio.

In [30]:
hybrid_records = []

for _, row in df_case5.iterrows():
    rec = build_hybrid_summary(row)
    rec.update({
        "doc_id": row["doc_id"],
        "title": row["title"],
        "summary_original": row["summary"],
        "text_clean_length_words": int(row["text_clean_length_words"]),
        "moncloa_section": row["moncloa_section"],
        "pdf_url": row["pdf_url"],
    })
    hybrid_records.append(rec)

df_summaries = pd.DataFrame(hybrid_records)

ordered_cols = [
    "doc_id", "title", "summary_original", "summary_seed_clean",
    "resumen_corto", "resumen_extendido",
    "summary_method", "summary_status", "summary_error",
    "n_evidence_sentences", "evidence_sentences_json",
    "text_clean_length_words", "moncloa_section", "pdf_url",
]
df_summaries = df_summaries[ordered_cols]

print(f"Resúmenes híbridos generados: {len(df_summaries)}")
display(df_summaries.head(3))

Resúmenes híbridos generados: 167


,doc_id,title,summary_original,summary_seed_clean,resumen_corto,resumen_extendido,summary_method,summary_status,summary_error,n_evidence_sentences,evidence_sentences_json,text_clean_length_words,moncloa_section,pdf_url
0,rtve_1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y cert...","El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y cert...","El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y cert...","El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y cert...",hybrid_summary_seed_plus_text_evidence,ok_hibrido_sin_llm,,7,"[""1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82"", ""La vista se interrumpió hasta las 10,00 horas del lunes día 22-02-82."", ""- El fiscal continúa pidiendo se lean decl...",640,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_20_de_febrero_de_1982.pdf
1,rtve_1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,"Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el Consejo Supremo de Justicia Militar, dond...","El documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el Consejo Supremo de Justicia Militar, donde se analizaron declaraciones y d...","El documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el Consejo Supremo de Justicia Militar, donde se analizaron declaraciones y d...","El documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el Consejo Supremo de Justicia Militar, donde se analizaron declaraciones y d...",hybrid_summary_seed_plus_text_evidence,ok_hibrido_sin_llm,,7,"[""1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 22-02-82."", ""- A las 12'09 el PRESIDENTE DEL CONSEJO suspende la vista."", ""- A las 12'35 se reanuda la vista."", ""- A las 14'05 ...",1018,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_22_de_febrero_de_1982.pdf
2,rtve_1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,"Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Justicia Militar en febrero de 1982, marca...","El documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Justicia Militar en febrero de 1982, marcadas por polémicas en torno a la c...","El documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Justicia Militar en febrero de 1982, marcadas por polémicas en torno a la c...","El documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Justicia Militar en febrero de 1982, marcadas por polémicas en torno a la c...",hybrid_summary_seed_plus_text_evidence,ok_hibrido_sin_llm,,7,"[""## 1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 23.02.82"", ""La última noticia es que la condición impuesta por los defensores es que tienen que suspender la acreditación a ...",1347,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_24_de_febrero_de_1982.pdf


**Lectura del bloque.**  
En este punto ya existe una salida válida para todos los documentos. Si la capa LLM no se usa, esta será la salida final; si se usa, solo sustituirá los documentos procesados correctamente.

## 6. Capa LLM opcional: reescritura controlada con evidencias

La capa LLM recibe únicamente:

- título;
- resumen original limpiado;
- evidencias seleccionadas desde `text_clean_base`.

No recibe información externa y debe devolver JSON con `resumen_corto` y `resumen_extendido`. Si la respuesta no es JSON válido o falla la API, se conserva el método híbrido.

Esta formulación evita enviar documentos completos muy largos y reduce el riesgo de límites de contexto. El LLM no se usa como fuente de conocimiento, sino como reescritor controlado a partir de evidencias.

In [7]:
def parse_json_response(text):
    text = str(text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("La respuesta no contiene JSON válido.")
    return json.loads(match.group(0))


def llm_chat_completion(messages, max_tokens=750, temperature=0.0, timeout=90):
    if not LLM_AVAILABLE:
        raise RuntimeError("LLM no disponible: falta activación explícita o token.")

    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "response_format": {"type": "json_object"},
    }

    request = urllib.request.Request(
        LLM_ENDPOINT,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {LLM_TOKEN}",
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8")
            data = json.loads(raw)
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code}: {body[:600]}") from exc

    return data["choices"][0]["message"]["content"]


def llm_rewrite_summary_from_evidence(row):
    evidence = json.loads(row["evidence_sentences_json"])
    evidence_block = "\n".join(f"- {e}" for e in evidence)

    system_prompt = (
        "Eres un asistente documental especializado en resumir documentos históricos. "
        "Debes usar únicamente el resumen semilla y las evidencias proporcionadas. "
        "No añadas contexto externo, no inventes nombres, fechas, cargos ni instituciones. "
        "Si una información no está en las evidencias, no la incluyas. "
        "Responde solo con JSON válido."
    )

    user_prompt = f"""
Título:
{row["title"]}

Resumen semilla procedente del corpus, probablemente truncado:
{row["summary_seed_clean"]}

Evidencias extraídas literalmente de text_clean_base:
{evidence_block}

Tarea:
Redacta una versión más completa, clara y legible del resumen documental.

Devuelve exactamente este JSON:
{{
  "resumen_corto": "máximo {SHORT_SUMMARY_WORDS} palabras",
  "resumen_extendido": "máximo {EXTENDED_SUMMARY_WORDS} palabras"
}}
""".strip()

    content = llm_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=800,
        temperature=0.0,
    )

    parsed = parse_json_response(content)

    resumen_corto = limit_words(parsed.get("resumen_corto", ""), SHORT_SUMMARY_WORDS)
    resumen_extendido = limit_words(parsed.get("resumen_extendido", ""), EXTENDED_SUMMARY_WORDS)

    if not resumen_corto or not resumen_extendido:
        raise ValueError("El JSON del modelo no contiene los campos esperados con contenido.")

    return {
        "resumen_corto": resumen_corto,
        "resumen_extendido": resumen_extendido,
    }

## 7. Selección de documentos para LLM

Se seleccionan los documentos sobre los que se intentará aplicar la capa LLM. El parámetro `LLM_MAX_DOCUMENTS` controla cuántos documentos se intentan y `LLM_SELECTION_STRATEGY` define la estrategia de selección.

En esta ejecución se ha configurado `LLM_MAX_DOCUMENTS=167` y `LLM_SELECTION_STRATEGY=first`, por lo que se intenta aplicar el LLM a todo el corpus. La tabla de salida permite comprobar si realmente se han seleccionado todos los grupos de longitud documental.

In [8]:
def assign_length_bucket(n_words):
    if n_words < 300:
        return "corto_<300"
    if n_words < 1000:
        return "medio_300_999"
    if n_words < 3000:
        return "largo_1000_2999"
    return "muy_largo_3000+"


df_final = df_summaries.copy()
df_final["length_bucket"] = df_final["text_clean_length_words"].apply(assign_length_bucket)
df_final["n_llm_chunks"] = 0
df_final["llm_attempted"] = False


def select_llm_indices(df_input, max_documents, strategy="sample_by_length"):
    if max_documents <= 0:
        return []

    if strategy == "first":
        return df_input.index[:max_documents].tolist()

    if strategy == "longest":
        return df_input.sort_values("text_clean_length_words", ascending=False).index[:max_documents].tolist()

    if strategy == "sample_by_length":
        buckets = ["corto_<300", "medio_300_999", "largo_1000_2999", "muy_largo_3000+"]
        per_bucket = max(1, math.ceil(max_documents / len(buckets)))
        selected = []
        for bucket in buckets:
            subset = df_input[df_input["length_bucket"] == bucket]
            if subset.empty:
                continue
            selected.extend(
                subset.sample(n=min(per_bucket, len(subset)), random_state=RANDOM_STATE).index.tolist()
            )
        return selected[:max_documents]

    raise ValueError("LLM_SELECTION_STRATEGY debe ser 'sample_by_length', 'longest' o 'first'.")


llm_indices = select_llm_indices(df_final, LLM_MAX_DOCUMENTS, LLM_SELECTION_STRATEGY)

selection_report = pd.DataFrame({
    "selected_for_llm": df_final.index.isin(llm_indices),
    "length_bucket": df_final["length_bucket"],
}).groupby(["length_bucket", "selected_for_llm"]).size().reset_index(name="n_documents")

print(f"Documentos seleccionados para intento LLM: {len(llm_indices)}")
display(selection_report)

Documentos seleccionados para intento LLM: 167


,length_bucket,selected_for_llm,n_documents
0,corto_<300,True,53
1,largo_1000_2999,True,45
2,medio_300_999,True,51
3,muy_largo_3000+,True,18


**Lectura del bloque.**  
La selección incluye los 167 documentos del corpus: 53 documentos cortos, 51 medios, 45 largos y 18 muy largos. Esto permite evaluar la capa LLM sobre todo el corpus, no solo sobre una muestra. La decisión metodológica sigue siendo segura porque el fallback híbrido queda disponible para cualquier documento que falle por límites de API, error de formato o respuesta no válida.

## 8. Ejecución LLM con fallback obligatorio

Se intenta reescribir solo los documentos seleccionados. Cualquier error queda registrado en `summary_error` y se conserva el resumen híbrido.

In [9]:
llm_attempts = 0
llm_success = 0
llm_failures = 0

if not LLM_AVAILABLE:
    print("LLM no activado o token no disponible. Se conserva la salida híbrida para todos los documentos.")
else:
    for idx in llm_indices:
        llm_attempts += 1
        df_final.loc[idx, "llm_attempted"] = True

        try:
            llm_result = llm_rewrite_summary_from_evidence(df_final.loc[idx])

            df_final.loc[idx, "resumen_corto"] = llm_result["resumen_corto"]
            df_final.loc[idx, "resumen_extendido"] = llm_result["resumen_extendido"]
            df_final.loc[idx, "summary_method"] = f"llm_{LLM_PROVIDER}_rewrite_with_hybrid_fallback"
            df_final.loc[idx, "summary_status"] = "ok_llm"
            df_final.loc[idx, "summary_error"] = ""
            df_final.loc[idx, "n_llm_chunks"] = 1
            llm_success += 1

        except Exception as exc:
            df_final.loc[idx, "summary_method"] = "hybrid_fallback_after_llm_error"
            df_final.loc[idx, "summary_status"] = "ok_hibrido_por_error_llm"
            df_final.loc[idx, "summary_error"] = str(exc)[:700]
            llm_failures += 1

        if LLM_SLEEP_SECONDS > 0:
            time.sleep(LLM_SLEEP_SECONDS)

llm_report = {
    "llm_available": LLM_AVAILABLE,
    "llm_provider": LLM_PROVIDER,
    "llm_model": LLM_MODEL,
    "llm_attempts": llm_attempts,
    "llm_success": llm_success,
    "llm_failures_with_fallback": llm_failures,
    "llm_outputs_total": int((df_final["summary_status"] == "ok_llm").sum()),
    "fallback_outputs_total": int((df_final["summary_status"] != "ok_llm").sum()),
}

pd.DataFrame([llm_report])

,llm_available,llm_provider,llm_model,llm_attempts,llm_success,llm_failures_with_fallback,llm_outputs_total,fallback_outputs_total
0,True,github,openai/gpt-4.1-mini,167,112,55,112,55


In [23]:
# Diagnóstico de errores de la capa LLM
error_diagnostics = df_final.copy()
error_diagnostics["error_type"] = np.select(
    [
        error_diagnostics["summary_status"].eq("ok_llm"),
        error_diagnostics["summary_error"].fillna("").str.contains("HTTP 429", regex=False),
        error_diagnostics["summary_error"].fillna("").str.contains("JSON", case=False, regex=True),
    ],
    [
        "sin_error_llm",
        "rate_limit_http_429",
        "respuesta_no_json_valida",
    ],
    default="otro_error_llm",
)

error_report = (
    error_diagnostics
    .groupby(["summary_status", "error_type"])
    .size()
    .reset_index(name="n_documents")
    .sort_values(["summary_status", "n_documents"], ascending=[True, False])
)

display(error_report)

,summary_status,error_type,n_documents
0,ok_hibrido_por_error_llm,rate_limit_http_429,54
1,ok_hibrido_por_error_llm,respuesta_no_json_valida,1
2,ok_llm,sin_error_llm,112


**Lectura del bloque.**  
La capa LLM se ha intentado sobre los 167 documentos del corpus. El resultado es parcialmente satisfactorio: 112 documentos terminan con `ok_llm` y 55 activan fallback híbrido. Por tanto, el notebook demuestra uso real del LLM a escala de corpus, pero también muestra una limitación práctica de la API: no todas las llamadas se completan correctamente. La salida sigue siendo válida porque los 55 errores no interrumpen el pipeline y quedan cubiertos por `hybrid_fallback_after_llm_error`.

## 9. Validación de salida final

Se comprueba que hay una fila por documento, que no se pierden identificadores y que los resúmenes no están vacíos.

In [24]:
final_validation = {
    "n_input_documents": len(df_case5),
    "n_output_rows": len(df_final),
    "n_unique_doc_id_output": df_final["doc_id"].nunique(),
    "n_empty_resumen_corto": int(df_final["resumen_corto"].fillna("").str.strip().eq("").sum()),
    "n_empty_resumen_extendido": int(df_final["resumen_extendido"].fillna("").str.strip().eq("").sum()),
    "n_missing_pdf_url": int(df_final["pdf_url"].fillna("").str.strip().eq("").sum()),
    "n_llm_ok": int((df_final["summary_status"] == "ok_llm").sum()),
    "n_llm_errors_with_fallback": int((df_final["summary_status"] == "ok_hibrido_por_error_llm").sum()),
}

pd.DataFrame([final_validation])

,n_input_documents,n_output_rows,n_unique_doc_id_output,n_empty_resumen_corto,n_empty_resumen_extendido,n_missing_pdf_url,n_llm_ok,n_llm_errors_with_fallback
0,167,167,167,0,0,0,112,55


**Lectura del bloque.**  
La validación confirma que se mantiene una salida completa: 167 filas, 167 `doc_id` únicos, ningún resumen vacío y ningún enlace PDF perdido. Además, separa claramente cobertura generativa y fallback: 112 documentos han sido reescritos con LLM y 55 han quedado cubiertos por fallback tras error. Esta distinción es clave para no presentar todo el corpus como generado por LLM.

In [25]:
assert len(df_final) == len(df_case5), "La salida final no conserva una fila por documento."
assert df_final["doc_id"].nunique() == df_case5["doc_id"].nunique(), "Se ha perdido trazabilidad por doc_id."
assert df_final["resumen_corto"].fillna("").str.strip().ne("").all(), "Hay resúmenes cortos vacíos."
assert df_final["resumen_extendido"].fillna("").str.strip().ne("").all(), "Hay resúmenes extendidos vacíos."

print("Validación superada: la salida mantiene trazabilidad y resúmenes no vacíos.")

Validación superada: la salida mantiene trazabilidad y resúmenes no vacíos.


## 10. Diagnóstico de puntos suspensivos

Se revisa si los puntos suspensivos siguen apareciendo como truncamiento real en las columnas finales. Esta comprobación es importante porque el `summary_original` estaba sistemáticamente recortado, mientras que las salidas finales pueden mostrar `...` por dos motivos distintos: truncamiento real de la fuente original o límite de longitud aplicado por el propio notebook.

In [26]:
# Diagnóstico de puntos suspensivos reales en las salidas
ellipsis_report = []

for col in ["summary_original", "summary_seed_clean", "resumen_corto", "resumen_extendido"]:
    values = df_final[col].fillna("").astype(str).str.strip()
    ellipsis_report.append({
        "column": col,
        "n_contains_ellipsis": int(values.str.contains(r"\.\.\.", regex=True).sum()),
        "n_ends_with_ellipsis": int(values.str.endswith("...").sum()),
    })

ellipsis_report = pd.DataFrame(ellipsis_report)
display(ellipsis_report)

ellipsis_by_status = (
    df_final.assign(
        resumen_extendido_contains_ellipsis=df_final["resumen_extendido"]
        .fillna("")
        .astype(str)
        .str.contains(r"\.\.\.", regex=True),
        resumen_extendido_ends_ellipsis=df_final["resumen_extendido"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.endswith("..."),
    )
    .groupby("summary_status")[[
        "resumen_extendido_contains_ellipsis",
        "resumen_extendido_ends_ellipsis",
    ]]
    .sum()
    .reset_index()
)

display(ellipsis_by_status)

,column,n_contains_ellipsis,n_ends_with_ellipsis
0,summary_original,167,167
1,summary_seed_clean,0,0
2,resumen_corto,4,4
3,resumen_extendido,23,23


,summary_status,resumen_extendido_contains_ellipsis,resumen_extendido_ends_ellipsis
0,ok_hibrido_por_error_llm,23,23
1,ok_llm,0,0


**Lectura del bloque.**  
El resumen original mantiene puntos suspensivos en los 167 documentos, lo que confirma que procede de una fuente truncada. La columna `summary_seed_clean` elimina correctamente ese truncamiento antes de usar el resumen como semilla. En la salida final, los resúmenes generados con LLM no contienen ni terminan en puntos suspensivos, mientras que algunos resúmenes de fallback híbrido sí lo hacen: 4 casos en `resumen_corto` y 23 en `resumen_extendido`. Esto no implica pérdida de trazabilidad, porque esos textos siguen procediendo de `summary` y de evidencias de `text_clean_base`, pero sí muestra una limitación del método híbrido cuando debe recortar el texto con `limit_words()`. Por tanto, los puntos suspensivos restantes deben interpretarse como efecto del fallback y del límite de longitud, no como el mismo truncamiento sistemático del resumen original.

## 11. Muestra para revisión manual

La evaluación automática de resúmenes es limitada. Se genera una muestra por grupos de longitud para revisar fidelidad, legibilidad y posible información no respaldada.

In [29]:
# Aseguramos que length_bucket existe aunque df_final haya sido reducido a final_cols
if "length_bucket" not in df_final.columns:
    df_final = df_final.copy()
    df_final["length_bucket"] = df_final["text_clean_length_words"].apply(assign_length_bucket)

# Muestra completa para revisión/exportación: 2 documentos por grupo de longitud
manual_review = (
    df_final
    .groupby("length_bucket", group_keys=False)
    .apply(lambda x: x.sample(n=min(2, len(x)), random_state=RANDOM_STATE))
    [[
        "length_bucket", "doc_id", "title", "summary_original", "summary_seed_clean",
        "resumen_corto", "resumen_extendido", "summary_method", "summary_status",
        "summary_error", "n_evidence_sentences", "text_clean_length_words", "pdf_url"
    ]]
    .sort_values(["length_bucket", "text_clean_length_words"])
    .reset_index(drop=True)
)

# Output compacto 1: composición de la muestra
manual_review_report = (
    manual_review
    .groupby(["length_bucket", "summary_status"])
    .size()
    .reset_index(name="n_documents")
)

display(manual_review_report)


def preview_text(text, max_chars=320):
    """Recorta texto solo para visualización en notebook."""
    text = normalize_spaces(str(text))
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."


# Seleccionamos un ejemplo LLM y un ejemplo fallback para comparar
example_llm = (
    manual_review[manual_review["summary_status"] == "ok_llm"]
    .head(1)
)

example_fallback = (
    manual_review[manual_review["summary_status"] == "ok_hibrido_por_error_llm"]
    .head(1)
)

manual_examples = pd.concat([example_llm, example_fallback], ignore_index=True)

example_cols = [
    "doc_id", "title", "summary_status", "summary_method",
    "summary_seed_clean", "resumen_corto", "resumen_extendido",
    "summary_error", "text_clean_length_words"
]

if not manual_examples.empty:
    manual_examples_compact = manual_examples[example_cols].copy()

    for col in ["summary_seed_clean", "resumen_corto", "resumen_extendido", "summary_error"]:
        manual_examples_compact[col] = manual_examples_compact[col].apply(
            lambda x: preview_text(x, max_chars=320)
        )

    display(manual_examples_compact)
else:
    print("No hay ejemplos disponibles para comparar LLM y fallback en la muestra seleccionada.")

,length_bucket,summary_status,n_documents
0,corto_<300,ok_hibrido_por_error_llm,1
1,corto_<300,ok_llm,1
2,largo_1000_2999,ok_hibrido_por_error_llm,1
3,largo_1000_2999,ok_llm,1
4,medio_300_999,ok_hibrido_por_error_llm,2
5,muy_largo_3000+,ok_llm,2


,doc_id,title,summary_status,summary_method,summary_seed_clean,resumen_corto,resumen_extendido,summary_error,text_clean_length_words
0,rtve_1726,RESERVADO: comunicación procesamiento implicado.,ok_llm,llm_github_rewrite_with_hybrid_fallback,"El documento es una comunicación oficial del Consejo Supremo de Justicia Militar, fechada el 10 de abril de 1981, dirigida al Ministro de Defensa por la Teniente General Presid...","El documento es una comunicación oficial del Consejo Supremo de Justicia Militar, fechada el 10 de abril de 1981, dirigida al Ministro de Defensa por la Teniente General Presid...","El documento constituye una comunicación oficial emitida por el Consejo Supremo de Justicia Militar el 10 de abril de 1981, dirigida al Ministro de Defensa por la Teniente Gene...",,154
1,rtve_1761,D.17._AGMAE_R40201_Exp._215,ok_hibrido_por_error_llm,hybrid_fallback_after_llm_error,"La página 1 presenta una comunicación oficial de la Cámara Municipal de Campo Maior dirigida al Cónsul de España en Elvas, fechada el 13 de marzo de 1921. En ella se expresa el...","La página 1 presenta una comunicación oficial de la Cámara Municipal de Campo Maior dirigida al Cónsul de España en Elvas, fechada el 13 de marzo de 1921. En ella se expresa el...","La página 1 presenta una comunicación oficial de la Cámara Municipal de Campo Maior dirigida al Cónsul de España en Elvas, fechada el 13 de marzo de 1921. En ella se expresa el...","HTTP 429: Too many requests. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-te...",169


**Lectura del bloque.**  
La muestra completa se conserva en `manual_review` para exportación, pero en el notebook solo se muestran dos salidas compactas: un ejemplo generado correctamente con LLM y un ejemplo resuelto mediante fallback. Esta comparación es suficiente para justificar la diferencia principal entre ambos métodos. En el caso `ok_llm`, el resumen aparece reescrito con mayor fluidez y menos dependencia literal de la semilla truncada. En el caso `ok_hibrido_por_error_llm`, la salida conserva más parecido con `summary_seed_clean`, porque la llamada a GitHub Models falló y se mantuvo el método híbrido. El error `HTTP 429` mostrado en el ejemplo fallback confirma que la limitación procede de la disponibilidad/límites de la API, no de una pérdida de trazabilidad del pipeline.

## 12. Exportación de resultados

Se guardan tres artefactos:

1. tabla final de resúmenes por documento;
2. resumen de métodos y estados;
3. muestra de revisión manual.

In [13]:
OUTPUT_SUMMARIES = OUTPUT_PROCESSED_DIR / "rtve_documentary_summaries_case5.csv"
OUTPUT_METHOD_REPORT = OUTPUT_TABLES_DIR / "caso_uso5_resumen_metodos.csv"
OUTPUT_STATUS_REPORT = OUTPUT_TABLES_DIR / "caso_uso5_resumen_estados.csv"
OUTPUT_MANUAL_REVIEW = OUTPUT_TABLES_DIR / "caso_uso5_muestra_revision_manual.csv"

final_cols = [
    "doc_id", "title", "summary_original", "summary_seed_clean",
    "resumen_corto", "resumen_extendido",
    "summary_method", "summary_status", "summary_error",
    "llm_attempted", "n_evidence_sentences", "evidence_sentences_json", "n_llm_chunks",
    "text_clean_length_words", "moncloa_section", "pdf_url",
]

df_final = df_final[final_cols]

df_final.to_csv(OUTPUT_SUMMARIES, index=False, encoding="utf-8")
manual_review.to_csv(OUTPUT_MANUAL_REVIEW, index=False, encoding="utf-8")

method_report = (
    df_final["summary_method"]
    .value_counts(dropna=False)
    .rename_axis("summary_method")
    .reset_index(name="n_documents")
)
status_report = (
    df_final["summary_status"]
    .value_counts(dropna=False)
    .rename_axis("summary_status")
    .reset_index(name="n_documents")
)

method_report.to_csv(OUTPUT_METHOD_REPORT, index=False, encoding="utf-8")
status_report.to_csv(OUTPUT_STATUS_REPORT, index=False, encoding="utf-8")

print("Archivos generados:")
print(f"- {OUTPUT_SUMMARIES.relative_to(PROJECT_ROOT)}")
print(f"- {OUTPUT_METHOD_REPORT.relative_to(PROJECT_ROOT)}")
print(f"- {OUTPUT_STATUS_REPORT.relative_to(PROJECT_ROOT)}")
print(f"- {OUTPUT_MANUAL_REVIEW.relative_to(PROJECT_ROOT)}")

display(method_report)
display(status_report)

Archivos generados:
- data/processed/rtve_documentary_summaries_case5.csv
- outputs/tables/caso_uso5_resumen_metodos.csv
- outputs/tables/caso_uso5_resumen_estados.csv
- outputs/tables/caso_uso5_muestra_revision_manual.csv


,summary_method,n_documents
0,llm_github_rewrite_with_hybrid_fallback,112
1,hybrid_fallback_after_llm_error,55


,summary_status,n_documents
0,ok_llm,112
1,ok_hibrido_por_error_llm,55


**Lectura del bloque.**  
Los informes exportados documentan la cobertura real del método. En esta ejecución, `caso_uso5_resumen_metodos.csv` muestra 112 documentos con `llm_github_rewrite_with_hybrid_fallback` y 55 con `hybrid_fallback_after_llm_error`. El informe de estados confirma la misma lectura: 112 `ok_llm` y 55 `ok_hibrido_por_error_llm`. Por tanto, el resultado final no debe describirse como 100 % generativo, sino como una generación asistida por LLM con fallback automático.

## 13. Figura auxiliar

Se genera una figura de apoyo para mostrar el método utilizado por grupo de longitud documental. Esta visualización ayuda a interpretar la cobertura del LLM y a detectar si los fallos se concentran en documentos cortos, medios, largos o muy largos.

In [14]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plot_df = df_final.copy()
    plot_df["length_bucket"] = plot_df["text_clean_length_words"].apply(assign_length_bucket)

    counts = (
        plot_df.groupby(["length_bucket", "summary_method"])
        .size()
        .unstack(fill_value=0)
        .reindex(["corto_<300", "medio_300_999", "largo_1000_2999", "muy_largo_3000+"])
    )

    ax = counts.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Mini caso 5 — Método de resumen por longitud documental")
    ax.set_xlabel("Grupo de longitud")
    ax.set_ylabel("Número de documentos")
    ax.legend(title="Método", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()

    fig_path = OUTPUT_FIGURES_DIR / "caso_uso5_metodo_por_longitud.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"Figura guardada en: {fig_path.relative_to(PROJECT_ROOT)}")

except Exception as exc:
    print(f"No se pudo generar la figura auxiliar, pero el pipeline principal no se ve afectado: {exc}")

Figura guardada en: outputs/figures/caso_uso5_metodo_por_longitud.png


## 14. Conclusiones del mini caso

El mini caso queda implementado como un pipeline reproducible de generación de resúmenes documentales para los 167 documentos del corpus.

Conclusiones técnicas:

* usa `text_clean_base` como fuente textual principal;
* conserva trazabilidad mediante `doc_id`, `title` y `pdf_url`;
* genera una salida completa para todos los documentos;
* parte del resumen original truncado como semilla, elimina su truncamiento inicial y lo completa con evidencias extraídas del texto limpio;
* intenta aplicar una capa LLM mediante GitHub Models al corpus completo;
* conserva automáticamente el fallback híbrido cuando la API falla, alcanza límites de uso o devuelve una respuesta no válida.

Resultado de esta ejecución:

* se intentó aplicar LLM a los 167 documentos;
* 112 documentos fueron reescritos correctamente con GitHub Models (`ok_llm`);
* 55 documentos activaron fallback híbrido tras error (`ok_hibrido_por_error_llm`);
* no se perdieron documentos, identificadores ni enlaces PDF;
* no quedaron resúmenes cortos ni extendidos vacíos;
* los resúmenes generados con LLM no presentan puntos suspensivos finales, mientras que algunos resúmenes híbridos sí los conservan por el límite de longitud aplicado en el fallback.

Limitaciones:

* el resultado no es 100 % generativo, porque 55 documentos quedaron cubiertos por fallback;
* los errores registrados se concentran principalmente en respuestas HTTP 429, por lo que la limitación principal no es estructural del notebook sino de disponibilidad/límites del servicio externo;
* el método híbrido es reproducible y trazable, pero puede parecerse bastante al resumen original porque no reescribe de forma generativa;
* algunos resúmenes híbridos pueden conservar puntos suspensivos finales cuando se recortan por límite de palabras;
* la revisión manual sigue siendo necesaria para comprobar fidelidad, legibilidad y ausencia de información no respaldada por las evidencias.

La conclusión defendible es que el notebook implementa una arquitectura robusta: intenta enriquecer el corpus completo con GitHub Models, documenta la cobertura real obtenida y garantiza una salida completa mediante fallback híbrido.
